In [ ]:
"""
End-to-end pipeline: CD133 E14 vs E18
DESeq2 -> Consensus Peaks -> Enhancer Integration -> CellOracle Base GRN -> GRN Pruning
"""

import os
import pandas as pd
import numpy as np
from importlib import reload
import pandas as pd 
import numpy as np 
import sys
import os
# get project root (two levels up from this notebook)
project_root = os.path.abspath(os.path.join(os.path.dirname('src'), '..'))
# if in notebook:
# project_root = os.path.abspath('..')   # or adjust as needed
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import src.pipeline as pipeline
import src.enhancer_atac as enatac
import src.grn_pruner as grnpruner
# Reload modules
reload(pipeline)
reload(enatac)
reload(grnpruner)

# ==============================================================================
# PATHS
# ==============================================================================

base_dir = "/home/users/adhal/CorticalNeuronFate/CellConversionNSC"

paths = {
    # RNA-seq
    "counts": os.path.join(base_dir, "data/rna_seq/E14_ E18_LGE_cortex_seq_Counts.csv"),
    "metadata": os.path.join(base_dir, "data/rna_seq/Samples+Pooling_RNA-Seq.csv"),
    
    # ATAC-seq
    "atac_metadata": os.path.join(base_dir, "data/atac_seq/ATAC-seq/samples_clean.csv"),
    "atac_peak_dir": os.path.join(base_dir, "data/atac_seq/ATAC-seq"),
    
    # Annotations
    "tf_list": os.path.join(base_dir, "data/annotations/Mouse_TFs_Kinases_webpage-3-30-2017.xlsx"),
    "gtf": os.path.join(base_dir, "data/annotations/gencode.vM36.annotation.gtf"),
    "enhancer_files": [
        os.path.join(base_dir, "data/annotations/enhancerAtlas_neuron_cortical.txt"),
        os.path.join(base_dir, "data/annotations/enhancerAtlas_brain_e14.5.txt"),
        os.path.join(base_dir, "data/annotations/enhancerAtlas_cortex.txt")
    ],
    
    # Integrated data (for pipeline object only)
    "overlap_df": os.path.join(base_dir, "data/integrated/overlap_annotated.tsv"),
    "chip_annotated": os.path.join(base_dir, "data/integrated/chip_annotated_filtered.tsv"),
    "atac_annotated": os.path.join(base_dir, "data/integrated/atac_annotated.tsv"),
    
    # Output paths
    "output_dir": os.path.join(base_dir, "results/cd133_e14_e18_grn"),
    "consensus_bed": os.path.join(base_dir, "results/cd133_e14_e18_grn/consensus_cd133.bed"),
    "deseq_output": os.path.join(base_dir, "results/cd133_e14_e18_grn/deseq_temporal_cd133.csv"),
    "celloracle_h5": os.path.join(base_dir, "results/cd133_e14_e18_grn/celloracle_tfinfo.h5"),
    "celloracle_parquet": os.path.join(base_dir, "results/cd133_e14_e18_grn/celloracle_base_grn.parquet"),
    "e14_grn": os.path.join(base_dir, "results/cd133_e14_e18_grn/E14_TF_network.csv"),
    "e18_grn": os.path.join(base_dir, "results/cd133_e14_e18_grn/E18_TF_network.csv")
}

# Create output directory
os.makedirs(paths["output_dir"], exist_ok=True)

exclude_samples = ['MUC9939', 'MUC9940', 'MUC9914']

print("="*80)
print("CD133 E14 vs E18 GRN PIPELINE")
print("="*80)

# ==============================================================================
# STEP 1: RNA-SEQ DIFFERENTIAL EXPRESSION (CD133 E14 vs E18)
# ==============================================================================

print("\n" + "="*80)
print("STEP 1: DIFFERENTIAL EXPRESSION ANALYSIS")
print("="*80)

nsc = pipeline.NSCAnalysis(
    counts_path=paths['counts'],
    metadata_path=paths['metadata'],
    atac_metadata_path=paths['atac_metadata'],
    overlap_df_path=paths['overlap_df'],
    chip_annotated_path=paths['chip_annotated'],
    atac_annotated_path=paths['atac_annotated'],
    tf_list_path=paths['tf_list'],
    gtf_path=paths['gtf'],
    exclude_samples=exclude_samples
)

# Filter for CD133 only
cd133_samples = nsc.metadata[nsc.metadata['Marker'] == 'Progenitors'].index.tolist()
nsc.counts = nsc.counts[[c for c in cd133_samples if c in nsc.counts.columns]]
nsc.metadata = nsc.metadata.loc[cd133_samples]

print(f"\nCD133 samples: {len(nsc.metadata)}")
print(nsc.metadata[['Stage', 'Region', 'Marker']].value_counts())

# Run DESeq2
deseq_results = nsc.run_deseq(group1='E14', group2='E18', group_col='Stage')

# Save DESeq results
deseq_results.to_csv(paths['deseq_output'])
print(f"\nDESeq results saved to: {paths['deseq_output']}")

# Get DE TFs
de_tfs = deseq_results[
    (deseq_results['padj'] < 0.05) & 
    (deseq_results['log2FoldChange'].abs() > 1) &
    (deseq_results['is_TF'] == True)
]

deg_list_e14 = de_tfs[de_tfs['log2FoldChange'] > 1]['symbol'].tolist()
deg_list_e18 = de_tfs[de_tfs['log2FoldChange'] < -1]['symbol'].tolist()

print(f"\nE14-high TFs: {len(deg_list_e14)}")
print(f"E18-high TFs: {len(deg_list_e18)}")
print(f"\nE14 TFs: {deg_list_e14[:10]}...")
print(f"E18 TFs: {deg_list_e18[:10]}...")

# ==============================================================================
# STEP 2: BUILD CONSENSUS PEAKS FROM ATAC-SEQ
# ==============================================================================

print("\n" + "="*80)
print("STEP 2: CONSENSUS PEAK BUILDING")
print("="*80)

builder = enatac.ConsensusPeakBuilder(
    metadata_file=paths['atac_metadata'],
    peak_dir=paths['atac_peak_dir'],
    factor='CD133',
    conditions=['E14', 'E18']
)

consensus = builder.build_consensus()
builder.save_bed(consensus, paths['consensus_bed'])

print(f"\nConsensus peaks: {len(consensus)}")
print(f"Saved to: {paths['consensus_bed']}")

# ==============================================================================
# STEP 3: ENHANCER INTEGRATION
# ==============================================================================

print("\n" + "="*80)
print("STEP 3: ENHANCER ATLAS INTEGRATION")
print("="*80)

integrator = enatac.EnhancerIntegrator(
    enhancer_files=paths['enhancer_files'],
    from_assembly='mm9'
)

enhancers_mm39 = integrator.load_and_liftover()

# Overlap with consensus peaks
target_genes, overlaps_df = integrator.overlap_with_consensus(consensus)

# Get DE TFs with accessible enhancers
e14_tfs_with_enh = integrator.get_de_with_accessible_enhancers(target_genes, deg_list_e14)
e18_tfs_with_enh = integrator.get_de_with_accessible_enhancers(target_genes, deg_list_e18)

print(f"\nE14 TFs with accessible enhancers: {e14_tfs_with_enh}")
print(f"E18 TFs with accessible enhancers: {e18_tfs_with_enh}")

# Save enhancer overlaps
overlaps_df.to_csv(os.path.join(paths['output_dir'], 'enhancer_consensus_overlaps.csv'), index=False)



/home/users/adhal/micromamba/envs/scrna_target_idf/lib/python3.10/site-packages/sorted_nearest/__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


CD133 E14 vs E18 GRN PIPELINE

STEP 1: DIFFERENTIAL EXPRESSION ANALYSIS


FileNotFoundError: counts: /home/users/adhal/CorticalNeuronFate/CellConversionNSC/data/rna_seq/E14__E18_LGE_cortex_seq_Counts.csv

In [ ]:
# ==============================================================================
# STEP 4: CELLORACLE BASE GRN CONSTRUCTION
# ==============================================================================

print("\n" + "="*80)
print("STEP 4: CELLORACLE BASE GRN CONSTRUCTION")
print("="*80)

from src.base_grn import GRNCo

# Initialize CellOracle
co = GRNCo(
    bed_path=paths['consensus_bed'],
    ref_genome='mm39',
    genomes_dir=None
)

# Step 1: Load BED
print("\nLoading consensus peaks...")
co.load_bed()

# Step 2: Annotate TSS
print("Annotating TSS...")
co.annotate_tss()

# Step 3: Ensure genome
print("Checking genome installation...")
co.ensure_genome()

# Step 4: Scan motifs
print("Scanning TF motifs (this takes time)...")
co.scan_motifs(fpr=0.02, verbose=True)

# Step 5: Filter motifs
print("Filtering motifs...")
co.filter_motifs(score_threshold=10)

# Step 6: Save outputs
print("Saving CellOracle outputs...")
co.save_tfinfo(paths['celloracle_h5'])
base_grn_df = co.save_dataframe(paths['celloracle_parquet'])

print(f"\nBase GRN saved to: {paths['celloracle_parquet']}")
print(f"Base GRN shape: {base_grn_df.shape}")
print(f"TFs in base GRN: {base_grn_df.shape[1] - 2}")  # -2 for peak_id and gene columns

# ==============================================================================
# STEP 5: GRN PRUNING (CORRELATION + DE FILTERING)
# ==============================================================================

print("\n" + "="*80)
print("STEP 5: GRN PRUNING")
print("="*80)

# Get all TFs (E14 + E18 DE TFs)
all_tf_list = list(set(deg_list_e14 + deg_list_e18))
print(f"\nTotal DE TFs for pruning: {len(all_tf_list)}")

# Initialize pruner
pruner = grnpruner.GRNPruner(
    pkn_file=paths['celloracle_parquet'],
    counts_file=paths['counts'],
    metadata_file=paths['metadata'],
    deseq_file=paths['deseq_output'],
    tf_list=all_tf_list,
    lfc_threshold=1.0,
    corr_threshold=0.3,
    qval_threshold=0.05
)

# Build stage-specific networks
e14_grn, e18_grn = pruner.build_networks()

# Save networks
e14_grn.to_csv(paths['e14_grn'], index=False)
e18_grn.to_csv(paths['e18_grn'], index=False)

print(f"\nE14 network saved to: {paths['e14_grn']}")
print(f"E18 network saved to: {paths['e18_grn']}")

# ==============================================================================
# STEP 6: SUMMARY
# ==============================================================================

print("\n" + "="*80)
print("PIPELINE COMPLETE - SUMMARY")
print("="*80)

summary = pd.DataFrame({
    'Step': [
        '1. DESeq2',
        '2. Consensus Peaks',
        '3. Enhancer Integration',
        '4. CellOracle Base GRN',
        '5. E14 TF Network',
        '6. E18 TF Network'
    ],
    'Count': [
        f"{len(de_tfs)} DE TFs",
        f"{len(consensus)} peaks",
        f"{len(target_genes)} genes with accessible enhancers",
        f"{base_grn_df.shape[1] - 2} TFs in base GRN",
        f"{len(e14_grn)} edges, {e14_grn['TF'].nunique()} TFs",
        f"{len(e18_grn)} edges, {e18_grn['TF'].nunique()} TFs"
    ]
})

print(summary.to_markdown(index=False))

print("\n" + "="*80)
print("OUTPUT FILES:")
print("="*80)
for key, path in paths.items():
    if 'output' in key or any(x in key for x in ['consensus', 'deseq', 'celloracle', 'grn']):
        print(f"  {key}: {path}")

print("\n" + "="*80)
print("NETWORK COMPARISON:")
print("="*80)

e14_edges = set(zip(e14_grn['TF'], e14_grn['target']))
e18_edges = set(zip(e18_grn['TF'], e18_grn['target']))

shared_edges = e14_edges & e18_edges
e14_specific = e14_edges - e18_edges
e18_specific = e18_edges - e14_edges

print(f"\nShared edges: {len(shared_edges)}")
print(f"E14-specific edges: {len(e14_specific)}")
print(f"E18-specific edges: {len(e18_specific)}")

print("\nTop 10 E14 hub TFs:")
print(e14_grn['TF'].value_counts().head(10))

print("\nTop 10 E18 hub TFs:")
print(e18_grn['TF'].value_counts().head(10))